# Laplace near-field jump relations

This notebook studies the one-sided near-field evaluation of the Laplace single- and double-layer potentials. The objective is to verify directly whether singularity extraction reproduces the correct jump behavior and to compare it with naive Gaussian quadrature.

The finite element surface $\Gamma_h$ is treated as the geometry of the experiment. We select a point $\boldsymbol y_0$ strictly inside one surface element, where the element normal $\boldsymbol n_h$ is unambiguous, and approach it symmetrically from both sides:

$$
\boldsymbol x_\delta^{\rm out}=\boldsymbol y_0+\delta\boldsymbol n_h,
\qquad
\boldsymbol x_\delta^{\rm in}=\boldsymbol y_0-\delta\boldsymbol n_h.
$$

Both potentials are evaluated with fixed interpolated FE densities. The jump relations are therefore tested for these discrete functions on $\Gamma_h$ itself. This isolates the near-field and quadrature behavior: the experiment does not measure the error of approximating an analytical geometry or analytical densities by $\Gamma_h$, $m_h$, and $j_h$. Such approximation errors would require a separate refinement study.

This distinction is essential when reading the companion notebook `laplace_nearfield`. There, the total potential $S(j_h)+D(m_h)$ is compared with the analytical exterior solution and the target approaches a supporting surface vertex. That experiment combines layer contributions and can therefore contain cancellation, reinforcement, FE trace interpolation error, and reference error. The present jump test avoids that combined-potential interpretation: it tests the one-sided jump and continuity criteria directly for fixed discrete data at an interior point of one element. It consequently complements, but does not replace, the broader near-field accuracy study.

For the double-layer potential and the convention used here,

$$
D(m_h)(\boldsymbol x_\delta^{\rm out})
-D(m_h)(\boldsymbol x_\delta^{\rm in})
\longrightarrow m_h(\boldsymbol y_0).
$$

Thus, the relative double-layer jump defect is

$$
E_D(\delta)=
\frac{\left|D(m_h)(\boldsymbol x_\delta^{\rm out})-D(m_h)(\boldsymbol x_\delta^{\rm in})-m_h(\boldsymbol y_0)\right|}
{|m_h(\boldsymbol y_0)|}.
$$

The single-layer potential is continuous across $\Gamma_h$, hence

$$
S(j_h)(\boldsymbol x_\delta^{\rm out})
-S(j_h)(\boldsymbol x_\delta^{\rm in})
\longrightarrow0.
$$

We measure this with the normalized continuity defect

$$
E_S(\delta)=
\frac{\left|S(j_h)(\boldsymbol x_\delta^{\rm out})-S(j_h)(\boldsymbol x_\delta^{\rm in})\right|}
{\max\{|S(j_h)(\boldsymbol x_\delta^{\rm out})|,|S(j_h)(\boldsymbol x_\delta^{\rm in})|\}}.
$$

A star denotes singularity extraction and a superscript $G$ denotes naive Gaussian quadrature. The exact jump criteria remain independent of a numerical reference. In addition, a singularity-extraction reference with `bonus_intorder=30` compares the four potential values at the same finite distance. It measures differences between quadratures, but can share an implementation bias with the lower-order evaluations. A jump defect at finite distance also includes the physical approach-to-the-limit contribution.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import Markdown, display
import numpy as np
import pandas as pd
from netgen.occ import Box, OCCGeometry, Pnt, Sphere
from ngsolve import BND, TRIG, CF, GridFunction, H1, Integrate, Mesh, SurfaceL2, TaskManager, ds, specialcf, sqrt, x, y, z
from ngsolve.fem import IntegrationRule
from ngsolve.bem import LaplaceDL, LaplaceSL

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FOUR_PI = 4.0 * np.pi
radius = 1.0
maxh = 0.2
curve_order = 3  # configurable; fixed during one experiment run
trace_order = 3
flux_order = trace_order - 1
source_point = np.array((0.20, -0.15, 0.10), dtype=float)
desired_direction = np.array((1.0, 1.0, 1.0), dtype=float)
desired_direction /= np.linalg.norm(desired_direction)
distances = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
bonus_intorders = list(range(5, 22, 2))
selected_bonus_intorder = bonus_intorders[-1]
reference_bonus_intorder = 30
reference_check_bonuses = [40, 60]
bem_params = {'use_fmm': False}

## Experiment profile



In [ ]:
display(Markdown(rf"""

| Quantity | Configuration in this experiment |
|---|---|
| FE geometry $\Gamma_h$ | fixed within one run: `maxh = {maxh}` |
| Geometry order | fixed within one run: `curve_order = {curve_order}`; configurable before a run |
| Pole $\boldsymbol x_s$ | fixed at `{tuple(float(value) for value in source_point)}` |
| Element search direction | fixed at `{tuple(float(value) for value in desired_direction)}`; configurable before a run |
| Trace spaces | fixed within one run: `H1(order={trace_order})`, `SurfaceL2(order={flux_order})` |
| Cauchy data | fixed interpolants $m_h=\Pi_h^1m$ and $j_h=\Pi_h^0j$ |
| Approach sides | inside and outside, evaluated symmetrically |
| Quadrature method | varied: naive Gauss ($G$) versus singularity extraction ($*$) |
| Quadrature bonuses | varied: `{bonus_intorders}` |
| Effective SL / DL orders | `2 + flux_order + bonus` / `2 + trace_order + bonus`, for both methods |
| Approach distances | varied: `{distances}` |
| Bonus shown in distance plot | `selected_bonus_intorder = {selected_bonus_intorder}` |
| Numerical reference | singularity extraction, `bonus_intorder={reference_bonus_intorder}` |
| Reference checks | additional bonuses `{reference_check_bonuses}`; separate jump criteria remain essential |
"""))

In [ ]:
sphere = Sphere((0, 0, 0), radius)
sphere.faces.name = 'sphere'
source_mesh = Mesh(OCCGeometry(sphere).GenerateMesh(maxh=maxh)).Curve(curve_order)
source_region = source_mesh.Boundaries('sphere')

boundary_elements = [element for element in source_mesh.Elements(BND) if len(element.vertices) == 3]
centroid_rule = IntegrationRule(TRIG, 1)  # one point at (1/3, 1/3)
mapped_centroids = source_mesh.MapToAllElements(centroid_rule, BND)
mapped_coordinates = np.column_stack((
    np.asarray(x(mapped_centroids)).reshape(-1),
    np.asarray(y(mapped_centroids)).reshape(-1),
    np.asarray(z(mapped_centroids)).reshape(-1),
))
if len(boundary_elements) != len(mapped_coordinates):
    raise ValueError('the current experiment expects a purely triangular surface mesh')

selected_index = max(
    range(len(boundary_elements)),
    key=lambda index: np.dot(
        mapped_coordinates[index] / np.linalg.norm(mapped_coordinates[index]),
        desired_direction,
    ),
)
selected_element = boundary_elements[selected_index]
selected_vertices = np.array(
    [source_mesh[vertex].point for vertex in selected_element.vertices],
    dtype=float,
)
surface_mesh_point = mapped_centroids[selected_index]
surface_point = np.array(
    (x(surface_mesh_point), y(surface_mesh_point), z(surface_mesh_point)),
    dtype=float,
)
normal_h = np.array(specialcf.normal(3)(surface_mesh_point), dtype=float)
normal_h /= np.linalg.norm(normal_h)
if np.dot(normal_h, surface_point) < 0:
    normal_h *= -1.0

boundary_triangles = [
    (element, np.array([source_mesh[vertex].point for vertex in element.vertices], dtype=float), mapped_coordinates[index])
    for index, element in enumerate(boundary_elements)
]
print(f'selected boundary element: {selected_element.nr}')
print(f'element-interior point: {tuple(surface_point)}')
print(f'outward element normal: {tuple(normal_h)}')


In [ ]:
distance_to_source = sqrt(
    (x-source_point[0])**2 + (y-source_point[1])**2 + (z-source_point[2])**2
)
density_seed = 1.0 / distance_to_source
gradient_seed = CF((density_seed.Diff(x), density_seed.Diff(y), density_seed.Diff(z)))
mesh_normal = specialcf.normal(3)

trace_space = H1(source_mesh, order=trace_order, definedon=source_region)
flux_space = SurfaceL2(source_mesh, order=flux_order, dual_mapping=False, definedon=source_region)
m_h = GridFunction(trace_space, name='m_h')
j_h = GridFunction(flux_space, name='j_h')
with TaskManager():
    m_h.Interpolate(density_seed, definedon=source_region)
    j_h.Interpolate(-(gradient_seed * mesh_normal), definedon=source_region)

m_at_surface = float(m_h(surface_mesh_point))
if abs(m_at_surface) < 1e-14:
    raise ValueError('m_h is too small at the selected point for a relative jump test')
print(f'm_h(y0) = {m_at_surface:.12e}')


## Symmetric approach points

The selected triangle is highlighted in red. The inner and outer points shown below use a visible representative distance; the dashed segment follows the normal of the mapped FE element at $\boldsymbol y_0$. Here $\boldsymbol y_0$ is the image of the barycentric point of the reference triangle under the element map, and therefore lies strictly inside the element. The background triangulation is a schematic rendering through the element vertices. The figure is saved as `output/laplace_jump_approach_points.png` for later use in the documentation.

In [ ]:
visualization_delta = 5e-2
point_out_vis = surface_point + visualization_delta * normal_h
point_in_vis = surface_point - visualization_delta * normal_h
all_triangles = [vertices for _, vertices, _ in boundary_triangles]

fig = plt.figure(figsize=(7.2, 6.2))
ax = fig.add_subplot(111, projection='3d')
ax.add_collection3d(Poly3DCollection(all_triangles, facecolor='lightsteelblue', edgecolor='gray', linewidth=0.25, alpha=0.22))
ax.add_collection3d(Poly3DCollection([selected_vertices], facecolor='crimson', edgecolor='darkred', linewidth=1.2, alpha=0.8))
ax.plot(*np.column_stack((point_in_vis, point_out_vis)), color='black', linestyle='--', linewidth=1.3, label='element normal')
ax.scatter(*surface_point, color='black', s=35, label=r'$y_0$')
ax.scatter(*point_out_vis, color='tab:green', s=55, label=r'$x_\delta^{out}$')
ax.scatter(*point_in_vis, color='tab:orange', s=55, label=r'$x_\delta^{in}$')
ax.set_xlim(surface_point[0]-0.25, surface_point[0]+0.25)
ax.set_ylim(surface_point[1]-0.25, surface_point[1]+0.25)
ax.set_zlim(surface_point[2]-0.25, surface_point[2]+0.25)
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_zlabel('$z$')
ax.set_title(
    f'Symmetric approach'
)
ax.legend(loc='best')
fig.tight_layout()
plt.savefig(OUTPUT_DIR / "laplace_jump_approach_points.png", dpi=300)
plt.show()


In [ ]:

def make_target(point, delta):
    target_mesh_size = max(0.1 * delta, 1e-7)
    box = Box(Pnt(*(point-target_mesh_size)), Pnt(*(point+target_mesh_size)))
    mesh = Mesh(OCCGeometry(box).GenerateMesh(maxh=target_mesh_size))
    return mesh, mesh(*point)


def evaluate_star(point_out, point_in, delta, bonus_intorder):
    mesh_out, mesh_point_out = make_target(point_out, delta)
    mesh_in, mesh_point_in = make_target(point_in, delta)
    m_trial = trace_space.TrialFunction()
    j_trial = flux_space.TrialFunction()
    with TaskManager():
        sl = LaplaceSL(j_trial * ds(bonus_intorder=bonus_intorder), **bem_params)(j_h)
        dl = LaplaceDL(m_trial * ds(bonus_intorder=bonus_intorder), **bem_params)(m_h)
        return {
            'sl_out': float(sl(mesh_point_out)), 'sl_in': float(sl(mesh_point_in)),
            'dl_out': float(dl(mesh_point_out)), 'dl_in': float(dl(mesh_point_in)),
        }


def evaluate_naive(point):
    tx, ty, tz = point
    dx, dy, dz = tx-x, ty-y, tz-z
    distance = sqrt(dx**2 + dy**2 + dz**2)
    sl_integrand = j_h / (FOUR_PI * distance)
    dl_kernel = (mesh_normal[0]*dx + mesh_normal[1]*dy + mesh_normal[2]*dz) / (FOUR_PI * distance**3)
    return sl_integrand, dl_kernel*m_h


def evaluate_naive_pair(point_out, point_in, bonus_intorder):
    sl_out_cf, dl_out_cf = evaluate_naive(point_out)
    sl_in_cf, dl_in_cf = evaluate_naive(point_in)
    # According to Numerical_Integration_Guide, section 1 (LinearForm):
    # q = 2*p_v + b, where p_v is the test function order and b is bonus_intorder
    q_sl = 2*flux_order + bonus_intorder  # for single-layer potential
    q_dl = 2*trace_order + bonus_intorder  # for double-layer potential
    with TaskManager():
        return {
            'sl_out': float(Integrate(sl_out_cf, source_mesh, definedon=source_region, order=q_sl)),
            'sl_in': float(Integrate(sl_in_cf, source_mesh, definedon=source_region, order=q_sl)),
            'dl_out': float(Integrate(dl_out_cf, source_mesh, definedon=source_region, order=q_dl)),
            'dl_in': float(Integrate(dl_in_cf, source_mesh, definedon=source_region, order=q_dl)),
        }


def defects(values):
    double_jump = values['dl_out'] - values['dl_in']
    single_jump = values['sl_out'] - values['sl_in']
    single_scale = max(abs(values['sl_out']), abs(values['sl_in']), 1e-15)
    return (
        abs(double_jump-m_at_surface) / abs(m_at_surface),
        abs(single_jump) / single_scale,
        double_jump, single_jump,
    )


## Jump experiment

The geometry, selected element, FE densities, and trace-space orders remain fixed. We vary only the side of approach, the distance $\delta$, the quadrature bonus, and the quadrature method. For fixed order, naive Gauss is expected to lose the local jump contribution as $\delta$ decreases. Singularity extraction should reproduce the double-layer jump and the continuity of the single layer down to small distances if the tangent projection and the remainder integration are accurate. The diagnostics below test this condition explicitly.

For every distance, the bonus-30 reference uses the same FE densities and targets. We retain the raw one-sided values, signed jumps, reference jump defects, and componentwise differences to that reference. All reference differences are normalized by `abs(m_h(y0))`, including the SL differences, so a small double-layer exterior value cannot cause a misleading relative error. The exact SL continuity criterion retains its original normalization.

We study the accuracy of the numerical integration scheme by systematic increase of `bonus_intorder`. Both methods now use the same effective order **for each layer potential**: `q_SL=2+flux_order+b`, `q_DL=2+trace_order+b`. The x-axis is the bonus `b`, not an absolute Gauss order.

In [ ]:
rows = []
reference_rows = []
reference_values = {}
for delta in distances:
    point_out = surface_point + delta*normal_h
    point_in = surface_point - delta*normal_h
    ref = evaluate_star(point_out, point_in, delta, reference_bonus_intorder)
    reference_values[delta] = ref
    rd, rs, rjump, rsjump = defects(ref)
    reference_rows.append({
        'delta': delta, 'bonus_intorder': reference_bonus_intorder,
        'double_layer_jump_defect': rd, 'single_layer_continuity_defect': rs,
        'computed_double_layer_jump': rjump, 'computed_single_layer_jump': rsjump,
        **ref,
    })
    for bonus in bonus_intorders:
        for method, values in (
            ('singularity extraction (*)', evaluate_star(point_out, point_in, delta, bonus)),
            ('naive Gauss (G)', evaluate_naive_pair(point_out, point_in, bonus)),
        ):
            double_defect, single_defect, double_jump, single_jump = defects(values)
            component_errors = {key+'_reference_error': abs(values[key]-ref[key])/abs(m_at_surface)
                                for key in ref}
            rows.append({
                'method': method, 'delta': delta, 'bonus_intorder': bonus,
                'double_layer_jump_defect': double_defect,
                'single_layer_continuity_defect': single_defect,
                'computed_double_layer_jump': double_jump,
                'exact_double_layer_jump': m_at_surface,
                'computed_single_layer_jump': single_jump,
                'dl_jump_reference_error': abs(double_jump-rjump)/abs(m_at_surface),
                'sl_jump_reference_error': abs(single_jump-rsjump)/abs(m_at_surface),
                'max_component_reference_error': max(component_errors.values()),
                **component_errors, **values,
            })

jump_results = pd.DataFrame(rows)
reference_results = pd.DataFrame(reference_rows)
jump_results.to_csv(OUTPUT_DIR / "laplace_jump_results.csv", index=False)
reference_results.to_csv(OUTPUT_DIR / "laplace_jump_reference.csv", index=False)
display(jump_results)

In [ ]:
styles = {
    'naive Gauss (G)': ('s', '--', 'tab:orange', 'naive Gauss'),
    'singularity extraction (*)': ('o', '-', 'tab:blue', 'singularity extraction'),
}
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.7))
for method, (marker, linestyle, color, label) in styles.items():
    data = jump_results[(jump_results.method == method) & (jump_results.bonus_intorder == selected_bonus_intorder)].sort_values('delta')
    axes[0].loglog(data.delta, data.double_layer_jump_defect, marker=marker, linestyle=linestyle, color=color, label=label)
    axes[1].loglog(data.delta, data.single_layer_continuity_defect, marker=marker, linestyle=linestyle, color=color, label=label)
ref_data = reference_results.sort_values('delta')
for ax, column in zip(axes, ['double_layer_jump_defect', 'single_layer_continuity_defect']):
    ax.loglog(ref_data.delta, ref_data[column], color='black', marker='x', linestyle=':',
              label=f'reference: bonus {reference_bonus_intorder}')
for ax in axes:
    ax.set_xlabel(r'$\delta$')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(loc='best')
    ax.invert_xaxis()
axes[0].set_ylabel('relative defect')
axes[0].set_title('Double-layer jump defect')
axes[1].set_title('Single-layer continuity defect')
fig.suptitle(f'One-sided limits | bonus={selected_bonus_intorder}, geometry order={curve_order}, H1={trace_order}, SurfaceL2={flux_order}')
fig.tight_layout()
plt.savefig(OUTPUT_DIR / "laplace_jump_defects_vs_distance.png", dpi=300)
plt.show()

closest_distance = min(distances)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.7))
for method, (marker, linestyle, color, label) in styles.items():
    data = jump_results[(jump_results.method == method) & (jump_results.delta == closest_distance)].sort_values('bonus_intorder')
    axes[0].semilogy(data.bonus_intorder, data.double_layer_jump_defect, marker=marker, linestyle=linestyle, color=color, label=label)
    axes[1].semilogy(data.bonus_intorder, data.single_layer_continuity_defect, marker=marker, linestyle=linestyle, color=color, label=label)
ref_closest = reference_results[reference_results.delta == closest_distance].iloc[0]
for ax, column in zip(axes, ['double_layer_jump_defect', 'single_layer_continuity_defect']):
    ax.axhline(ref_closest[column], color='black', linestyle=':',
               label=f'reference: bonus {reference_bonus_intorder}')
for ax in axes:
    ax.set_xlabel('bonus_intorder (matched effective SL / DL orders)')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(loc='best')
axes[0].set_ylabel('relative defect')
axes[0].set_title('Double-layer jump defect')
axes[1].set_title('Single-layer continuity defect')
fig.suptitle(rf'Quadrature comparison at $\delta={closest_distance:.0e}$ | geometry order={curve_order}, H1={trace_order}, SurfaceL2={flux_order}')
fig.tight_layout()
plt.savefig(OUTPUT_DIR / "laplace_jump_defects_vs_order.png", dpi=300)
plt.show()

## Reading the results

The distance plot separates two effects. As $\delta$ decreases, the exact jump relations predict convergence of the double-layer defect to zero and continuity of the single-layer defect to zero, up to roundoff and unresolved near-field integration. The reference curve still contains the finite-distance approach error, so it is not an exact zero-error baseline.

At the smallest distance, the bonus plot tests quadrature convergence at fixed geometry, densities, and target points. Increasing the bonus should reduce the quadrature component until the result is limited by the approach-to-the-limit error or floating-point effects. Singularity extraction is successful when it preserves the predicted jump and continuity where naive Gauss loses accuracy. The reference-error columns compare implementations at finite distance only; the jump defects are the primary diagnostics.